# DistilBERT + SQL features — fusion model (Phase 2)

**Bar to beat** (TF-IDF baseline on v3, `training/test.ipynb`): within one company **0.8115**, within
product × issue **0.9457**, pooled PR-AUC 0.4262.

**Runs:** set `RUN` in cell 1 — `"mac"` (512 rows, finds bugs) → `"smoke"` (Kaggle, 10k rows) → `"full"`.

| part | choice (CLAUDE.md §9, TriageIQ.md §2.3) |
|---|---|
| text | DistilBERT, 512 tokens, `[DATE]`/`[REDACTED]` as special tokens |
| features | 4 categories → embeddings · 15 numbers → standardized on train → MLP → 64 |
| fusion | text 768→128 + features 64 → head → logit |
| training | epoch 1 encoder frozen; then lr 2e-5 encoder / 1e-3 rest; ≤3 epochs; best val PR-AUC |
| speed | length-grouped batches, dynamic padding, fp16 on CUDA |

In [ ]:
# 1. config
import os, json, math, time, hashlib, random
from pathlib import Path
import numpy as np, pandas as pd, torch

RUN  = "smoke"            # "mac" | "smoke" | "full"
ON_KAGGLE = Path("/kaggle/input").exists()
if not ON_KAGGLE: RUN = "mac"
CFG = {
    "mac":   dict(n_train=512,   n_val=256,   n_test=512,    epochs=2, batch=8,  eval_batch=8),
    "smoke": dict(n_train=10000, n_val=5000,  n_test=10000,  epochs=2, batch=16, eval_batch=64),
    "full":  dict(n_train=None,  n_val=None,  n_test=None,   epochs=3, batch=16, eval_batch=64),
}[RUN]
CFG.update(model="distilbert-base-uncased", max_len=512, trunc="head",   # "head" | "head_tail"
           lr_enc=2e-5, lr_rest=1e-3, warmup=0.05, weight_decay=0.01, patience=1, seed=42)
SEED = CFG["seed"]; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
AMP = DEV == "cuda"                                   # fp16 only on the T4
OUT = Path("/kaggle/working" if ON_KAGGLE else "outputs") / f"fusion_{RUN}"
OUT.mkdir(parents=True, exist_ok=True)
print(RUN, "| device", DEV, "| fp16", AMP, "| out", OUT, "\n", CFG)

In [ ]:
# 2. load v3, check it is the exported file, take the run's subset
import glob
def find(name):
    roots = ["/kaggle/input"] if ON_KAGGLE else ["../Data/data/interim"]
    hits = [h for r in roots for h in glob.glob(f"{r}/**/{name}", recursive=True)]
    assert len(hits) == 1, f"{name}: {hits}"
    return hits[0]
manifest = json.load(open(find("triageiq_training_v3.manifest.json")))
PQ = find(manifest["file"])
assert hashlib.sha256(open(PQ, "rb").read()).hexdigest() == manifest["sha256"], "file differs from manifest"
df = pd.read_parquet(PQ)
def take(split, n):
    d = df[df.split == split]
    return (d if n is None or n >= len(d) else d.sample(n, random_state=SEED)).reset_index(drop=True)
tr, va, te = take("train", CFG["n_train"]), take("val", CFG["n_val"]), take("test", CFG["n_test"])
for name, d in [("train", tr), ("val", va), ("test", te)]:
    print(f"{name:<5} {len(d):>7,} rows · {d.paid.mean():.2%} payouts")

In [ ]:
# 3. tokenize — the two blank markers become ONE token each (else '[REDACTED]' = 5 pieces)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(CFG["model"])
tok.add_special_tokens({"additional_special_tokens": manifest["text_markers"]})
L = CFG["max_len"] - 2                                   # room for [CLS] and [SEP]
def encode(texts):
    body = tok(list(texts), add_special_tokens=False, truncation=False)["input_ids"]
    full_len = np.array([len(b) for b in body])
    if CFG["trunc"] == "head_tail":                          # keep the start and the end
        h = L // 4
        body = [b if len(b) <= L else b[:h] + b[-(L - h):] for b in body]
    else:
        body = [b[:L] for b in body]
    return [[tok.cls_token_id] + b + [tok.sep_token_id] for b in body], full_len
t0 = time.time()
for d in (tr, va, te):
    d["ids"], d["n_tokens"] = encode(d.text)
print(f"tokenized in {time.time() - t0:.0f}s · markers are single tokens:",
      [len(tok(m, add_special_tokens=False)['input_ids']) for m in manifest['text_markers']])
print(f"cut at 512 — train {np.mean(tr.n_tokens > L):.1%} · test {np.mean(te.n_tokens > L):.1%}")

In [ ]:
# 4. tabular inputs — every statistic is fit on TRAIN only and saved (guardrail §10)
CAT = manifest["columns"]["categorical"]
NUM = manifest["columns"]["flags"] + manifest["columns"]["numeric"]
LOG1P = ["company_quiet_days"]                            # heavy tail: days can run into the hundreds
def num_matrix(d):
    x = d[NUM].astype("float64").copy()
    x[LOG1P] = np.log1p(x[LOG1P])
    return x
mu, sd = num_matrix(tr).mean(), num_matrix(tr).std().replace(0, 1)
vocab = {c: int(df[c].max()) + 1 for c in CAT}           # ids are small integers; index = id
prep = {"num": NUM, "cat": CAT, "log1p": LOG1P, "mean": mu.to_dict(), "std": sd.to_dict(), "vocab": vocab,
        "max_len": CFG["max_len"], "trunc": CFG["trunc"], "markers": manifest["text_markers"]}
(OUT / "preprocessing.json").write_text(json.dumps(prep, indent=1))
for d in (tr, va, te):
    d["num"] = list(((num_matrix(d) - mu) / sd).to_numpy(np.float32))
    d["cat"] = list(d[CAT].to_numpy(np.int64))
print("numeric", len(NUM), "· categorical vocab", vocab)

In [ ]:
# 5. batches of similar length (group_by_length): 2.23x fewer padded tokens on this data
def batches(d, bs, shuffle):
    idx = np.arange(len(d))
    if shuffle:
        rng = np.random.default_rng(SEED + batches.calls); batches.calls += 1
        idx = rng.permutation(idx)
        chunks = [idx[i:i + 50 * bs] for i in range(0, len(idx), 50 * bs)]          # mega-batches
        idx = np.concatenate([c[np.argsort(d.n_tokens.values[c])] for c in chunks])
        out = [idx[i:i + bs] for i in range(0, len(idx), bs)]
        rng.shuffle(out)
        return out
    idx = idx[np.argsort(d.n_tokens.values)]                                          # eval: sorted
    return [idx[i:i + bs] for i in range(0, len(idx), bs)]
batches.calls = 0
def collate(d, b):
    ids = [d.ids.iat[i] for i in b]; m = max(map(len, ids))
    x = torch.full((len(b), m), tok.pad_token_id); a = torch.zeros((len(b), m), dtype=torch.long)
    for j, s in enumerate(ids):
        x[j, :len(s)] = torch.tensor(s); a[j, :len(s)] = 1
    return (x.to(DEV), a.to(DEV), torch.tensor(np.stack([d.cat.iat[i] for i in b])).to(DEV),
            torch.tensor(np.stack([d.num.iat[i] for i in b])).to(DEV),
            torch.tensor(d.paid.values[b], dtype=torch.float32).to(DEV))

In [ ]:
# 6. the model: text branch + feature branch -> one logit
import torch.nn as nn
from transformers import AutoModel
EMB = {"product_id": 4, "sub_product_id": 8, "issue_id": 12, "state_id": 8}
class Fusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = AutoModel.from_pretrained(CFG["model"])
        self.enc.resize_token_embeddings(len(tok))                    # room for the 2 markers
        self.text = nn.Sequential(nn.Linear(self.enc.config.dim, 128), nn.GELU(), nn.Dropout(0.1))
        self.embs = nn.ModuleList([nn.Embedding(vocab[c], EMB[c]) for c in CAT])
        self.tab = nn.Sequential(nn.Linear(sum(EMB.values()) + len(NUM), 128), nn.GELU(), nn.Dropout(0.1),
                                 nn.Linear(128, 64), nn.GELU())
        self.head = nn.Sequential(nn.Linear(128 + 64, 64), nn.GELU(), nn.Dropout(0.1), nn.Linear(64, 1))
    def forward(self, ids, att, cat, num):
        t = self.text(self.enc(input_ids=ids, attention_mask=att).last_hidden_state[:, 0])   # [CLS]
        g = self.tab(torch.cat([e(cat[:, i]) for i, e in enumerate(self.embs)] + [num], 1))
        return self.head(torch.cat([t, g], 1)).squeeze(1)
model = Fusion().to(DEV)
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

In [ ]:
# 7. scoring — the same three scores as training/test.ipynb, plus the cut-at-512 slice
from sklearn.metrics import roc_auc_score, average_precision_score
def within(d, p, keys, min_pos, min_n):
    aucs, wts = [], []
    for _, idx in d.groupby(keys, observed=True).indices.items():
        y = d.paid.values[idx]
        if y.sum() < min_pos or y.sum() == len(y) or len(y) < min_n: continue
        aucs.append(roc_auc_score(y, p[idx])); wts.append(len(y))
    return (float(np.average(aucs, weights=wts)) if aucs else float("nan")), len(aucs)
@torch.no_grad()
def predict(d):
    model.eval(); out = np.zeros(len(d))
    for b in batches(d, CFG["eval_batch"], shuffle=False):   # Mac: 8 — 64 x 512 tokens ran out of memory
        ids, att, cat, num, _ = collate(d, b)
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            out[b] = model(ids, att, cat, num).float().cpu().numpy()
    return out                                                       # logits
def scores(d, logit):
    p, y = 1 / (1 + np.exp(-logit)), d.paid.values
    ws, ns = within(d, p, ["product_id", "issue_id"], 25, 150)
    wc, nc = within(d, p, ["company_id"], 20, 200)
    cut = d.n_tokens.values > L
    s = {"pooled_auc": roc_auc_score(y, p), "pooled_pr": average_precision_score(y, p),
         "within_strata_auc": ws, "n_strata": ns, "within_company_auc": wc, "n_companies": nc}
    for name, m in [("auc_fits_512", ~cut), ("auc_cut_at_512", cut)]:
        s[name] = roc_auc_score(y[m], p[m]) if 0 < y[m].sum() < m.sum() else float("nan")
    return s

In [ ]:
# 8. train: epoch 1 with DistilBERT frozen, then everything; keep the best val PR-AUC
from transformers import get_linear_schedule_with_warmup
enc_params = list(model.enc.parameters())
rest = [p for n, p in model.named_parameters() if not n.startswith("enc.")]
opt = torch.optim.AdamW([{"params": enc_params, "lr": CFG["lr_enc"]}, {"params": rest, "lr": CFG["lr_rest"]}],
                        weight_decay=CFG["weight_decay"])
steps = CFG["epochs"] * math.ceil(len(tr) / CFG["batch"])
sched = get_linear_schedule_with_warmup(opt, int(CFG["warmup"] * steps), steps)
scaler = torch.amp.GradScaler("cuda", enabled=AMP)
lossf = nn.BCEWithLogitsLoss()
best, bad, history = -1.0, 0, []
for ep in range(1, CFG["epochs"] + 1):
    frozen = ep == 1
    for p in enc_params: p.requires_grad = not frozen
    model.train(); t0, tot, n = time.time(), 0.0, 0
    for b in batches(tr, CFG["batch"], shuffle=True):
        ids, att, cat, num, y = collate(tr, b)
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            loss = lossf(model(ids, att, cat, num), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item() * len(b); n += len(b)
    v = scores(va, predict(va))
    history.append({"epoch": ep, "frozen": frozen, "train_loss": tot / n, "secs": time.time() - t0} | v)
    print(f"epoch {ep} {'(frozen)' if frozen else '         '} loss {tot / n:.4f} · val PR-AUC {v['pooled_pr']:.4f}"
          f" · AUC {v['pooled_auc']:.4f} · {time.time() - t0:.0f}s")
    if v["pooled_pr"] > best:
        best, bad = v["pooled_pr"], 0
        torch.save(model.state_dict(), OUT / "model.pt")
    else:
        bad += 1
        if bad > CFG["patience"]: print("early stop"); break

In [ ]:
# 9. test with the best epoch's weights + recalibration check
import gc
del opt, sched, scaler; gc.collect()                         # free the optimizer's GPU memory first
if DEV == "cuda": torch.cuda.empty_cache()
if DEV == "mps":  torch.mps.empty_cache()
model.load_state_dict(torch.load(OUT / "model.pt", map_location="cpu"))   # via CPU: no 2nd GPU copy
if RUN == "mac":                                              # Mac GPU memory is shared with Docker:
    DEV = "cpu"; model.to(DEV)                                #  test on the CPU (Kaggle keeps the T4)
logit = predict(te)
res = scores(te, logit)
off = manifest["recalibration"]["logit_offset"]               # case-control -> real-world odds
p_cal = 1 / (1 + np.exp(-(logit + off)))
res |= {"calibrated_mean_p": float(p_cal.mean()), "actual_rate": float(te.paid.mean())}
print(pd.Series(res).round(4).to_string())
print("\nbar to beat (TF-IDF, full v3 test): within-company 0.8115 · within-strata 0.9457 · PR 0.4262")
json.dump({"run": RUN, "cfg": CFG, "history": history, "test": res}, open(OUT / "metrics.json", "w"), indent=1, default=float)
pd.DataFrame({"complaint_id": te.complaint_id, "logit": logit, "p_cal": p_cal}).to_parquet(OUT / "test_predictions.parquet")
print("saved:", sorted(p.name for p in OUT.iterdir()))